In [ ]:
# parse_instruction_ver18 (修正GT での ver17 再検証)

## 背景
- GT の `annotations_gt_task_ver10.json` から `target` フィールドの `subject` プレースホルダーを削除した（44 箇所置換）。
- instruction から具体的な名詞を推定して置き換え（speaker, chef, face, turquoise logo など）。

## 目的
- 修正済み GT を使って ver17 の予測を再評価する。
- GT 変更による性能変化（スコア向上）を計測する。
- 汎用 `subject` プレースホルダーを削除することで、予測モデルへの目標値が明確化どう影響するかを観察。

## 方法
1. 修正済み GT: `annotations_gt_task_ver10.json`
2. ver17 予測: `unknown_predictions_v17d_multi.json`（single + multi）
3. 評価方法: ver17 と同じ（target 包含一致、action 完全一致、params 一致率）
4. 比較対象: 修正前（ver09）vs 修正後（ver10）の GT を用いた evaluation results

## 出力
- 修正前後でのスコア比較表
- 改善 / 低下した action 別統計


2


In [1]:
from __future__ import annotations

import copy
import json
import re
import sys
from collections import Counter, defaultdict
from difflib import SequenceMatcher
from pathlib import Path

WORKSPACE = Path('/workspace')
SRC_DIR = WORKSPACE / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from parse.data_loading import load_base_records, load_grouped_unknown_records

DATA_DIR     = WORKSPACE / 'data'
NOTEBOOK_DIR = WORKSPACE / 'notebook'
OUTPUT_DIR   = NOTEBOOK_DIR / 'ver18_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_PATH      = DATA_DIR / 'annotations.jsonl'
GT_PATH_OLD   = DATA_DIR / 'annotations_gt_task_ver09.json'
GT_PATH_NEW   = DATA_DIR / 'annotations_gt_task_ver10.json'  # subject削除済み
GROUPED_PATHS = [
    DATA_DIR / 'annotations_grouped_ver01.json',
    DATA_DIR / 'annotations_grouped_ver02.json',
]

# 修正済み GT で base_records を構築
base_records = load_base_records(RAW_PATH, GT_PATH_NEW)

# ver17 の unknown records の定義
unknown_records = load_grouped_unknown_records(
    GROUPED_PATHS, base_records, instruction_keys=('ver2', 'ver3', 'ver4')
)

print('base_records (GT_ver10 使用):', len(base_records))
print('unknown_records:', len(unknown_records))
print('output_dir:', OUTPUT_DIR)


base_records: 100
unknown_records: 600
output_dir: /workspace/notebook/ver18_outputs


## 1. ユーティリティ + 評価関数（v15d ロジック再実装）

In [16]:
# ---- text utilities ----

def normalize_text(x):
    if x is None:
        return ''
    t = str(x).lower().replace('_', ' ').strip()
    t = re.sub(r'[^a-z0-9\s]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()


def flatten_json(v, prefix=''):
    out = {}
    if isinstance(v, dict):
        for k, c in v.items():
            p = f'{prefix}.{k}' if prefix else str(k)
            out.update(flatten_json(c, p))
    elif isinstance(v, list):
        for i, c in enumerate(v):
            out.update(flatten_json(c, f'{prefix}[{i}]'))
    else:
        out[prefix] = normalize_text(v)
    return out


def text_similarity(a, b):
    aa, bb = normalize_text(a), normalize_text(b)
    if not aa and not bb:
        return 1.0
    if not aa or not bb:
        return 0.0
    j = len(set(aa.split()) & set(bb.split())) / max(1, len(set(aa.split()) | set(bb.split())))
    return 0.6 * j + 0.4 * SequenceMatcher(None, aa, bb).ratio()


def _target_values(target):
    return target if isinstance(target, list) else [target]


def contains_subject_token(target):
    return any(
        isinstance(v, str) and 'subject' in normalize_text(v)
        for v in _target_values(target)
    )


def target_text(target):
    vals = [normalize_text(v) for v in _target_values(target) if normalize_text(v)]
    return ' '.join(vals).strip()


def params_score(pred_params, gt_params):
    pp = flatten_json(pred_params)
    gp = flatten_json(gt_params)
    if not gp:
        return 1.0
    if not pp:
        return 0.0
    m = sum(1 for k, gv in gp.items() if k in pp and (pp[k] == gv or pp[k] in gv or gv in pp[k]))
    return m / len(gp)


def prediction_has_subject(pred):
    tasks = pred.get('tasks', []) if isinstance(pred, dict) else []
    return any(contains_subject_token(task.get('target', '')) for task in tasks)


# ---- strict single-task scorer (subject in any pred target => total 0) ----

def score_primary(pred, gt_primary):
    if prediction_has_subject(pred):
        return {
            'action_score': 0.0,
            'target_score': 0.0,
            'params_score': 0.0,
            'total': 0.0,
            'subject_invalid': 1.0,
        }

    task = pred.get('tasks', [{}])[0]
    action = 1.0 if task.get('action', '') == gt_primary.get('action', '') else 0.0
    pt = target_text(task.get('target', ''))
    gt = target_text(gt_primary.get('target', ''))
    target = 1.0 if (pt and gt and (pt in gt or gt in pt)) else 0.0
    params = params_score(task.get('params', {}), gt_primary.get('params', {}))
    total = 0.5 * action + 0.2 * target + 0.3 * params
    return {
        'action_score': round(action, 4),
        'target_score': round(target, 4),
        'params_score': round(params, 4),
        'total': round(total, 4),
        'subject_invalid': 0.0,
    }


def evaluate_records(records, predict_fn):
    rows, pred_map, dbg_map = [], {}, {}
    for r in records:
        pred, dbg = predict_fn(r)
        key = r['prediction_key']
        pred_map[key] = pred
        dbg_map[key] = dbg
        s = score_primary(pred, r['gt_primary'])
        rows.append({'prediction_key': key, 'video_path': r['video_path'], **s})
    overall = {
        k: round(sum(x[k] for x in rows) / len(rows), 4)
        for k in ['action_score', 'target_score', 'params_score', 'total', 'subject_invalid']
    }
    return {'rows': rows, 'overall': overall, 'predictions': pred_map, 'debug': dbg_map}


# ---- strict multi-task scorer (subject in any pred target => total 0) ----

def task_score_multi(pred_task, gt_task):
    action = 1.0 if pred_task.get('action', '') == gt_task.get('action', '') else 0.0
    pt = target_text(pred_task.get('target', ''))
    gt = target_text(gt_task.get('target', ''))
    target = 1.0 if (pt and gt and (pt in gt or gt in pt)) else 0.0
    params = params_score(pred_task.get('params', {}), gt_task.get('params', {}))
    return 0.5 * action + 0.2 * target + 0.3 * params


def score_multi(pred_tasks, gt_tasks):
    pred_tasks = pred_tasks or []
    gt_tasks = gt_tasks or []
    if not pred_tasks and not gt_tasks:
        return {'coverage': 1.0, 'precision': 1.0, 'count_alignment': 1.0, 'total': 1.0, 'subject_invalid': 0.0}
    if any(contains_subject_token(task.get('target', '')) for task in pred_tasks):
        return {'coverage': 0.0, 'precision': 0.0, 'count_alignment': 0.0, 'total': 0.0, 'subject_invalid': 1.0}
    coverage = sum(max((task_score_multi(p, g) for p in pred_tasks), default=0.0) for g in gt_tasks) / max(1, len(gt_tasks))
    precision = sum(max((task_score_multi(p, g) for g in gt_tasks), default=0.0) for p in pred_tasks) / max(1, len(pred_tasks))
    m = max(1, len(pred_tasks), len(gt_tasks))
    count_alignment = 1.0 - abs(len(pred_tasks) - len(gt_tasks)) / m
    total = 0.55 * coverage + 0.35 * precision + 0.10 * count_alignment
    return {
        'coverage': round(coverage, 4),
        'precision': round(precision, 4),
        'count_alignment': round(count_alignment, 4),
        'total': round(total, 4),
        'subject_invalid': 0.0,
    }


def evaluate_multi(records, predict_fn):
    rows, pred_map = [], {}
    for r in records:
        pred = predict_fn(r)
        key = r['prediction_key']
        pred_map[key] = pred
        s = score_multi(pred['tasks'], r['gt_tasks'])
        rows.append({'prediction_key': key, 'video_path': r['video_path'], **s})
    overall = {
        k: round(sum(x[k] for x in rows) / len(rows), 4)
        for k in ['coverage', 'precision', 'count_alignment', 'total', 'subject_invalid']
    }
    return {'rows': rows, 'overall': overall, 'predictions': pred_map}


print('utilities ready (record-level subject invalidation enabled)')

utilities ready (record-level subject invalidation enabled)


## 2. action 推論 + v15d target/params ロジック（共通部品）
- v15d のロジックをそのまま再実装し、v17 系の比較基準とする。

In [3]:
_STYLE_VOCAB = [
    'anime', 'cyberpunk', 'pixel art', 'ukiyo', 'ghibli', 'watercolor',
    'oil painting', 'comic style', 'woodblock', '16-bit', 'retro pixel',
    'aesthetic', 'painting style', 'art style', 'american comic',
]
_COLOR_WORDS = [
    'violet', 'blue', 'red', 'green', 'navy', 'emerald', 'burgundy', 'magenta',
    'orange', 'purple', 'pink', 'yellow', 'black', 'white', 'gray', 'grey',
    'silver', 'gold', 'teal', 'cyan', 'indigo', 'maroon', 'beige', 'coral',
    'turquoise', 'aqua', 'lime', 'lavender', 'brown', 'crimson', 'amber',
]

def infer_action(inst):
    t = inst.lower()
    if re.search(r'\barc\s+shot\b|\borbit\b|revolving\s+around', t): return 'orbit_camera'
    if re.search(r'increase\s+the\s+amount\s+of', t): return 'increase_amount'
    if re.search(r'increase\s+the\s+number\s+of', t): return 'add_object'
    if re.search(r'\bdolly[- ]?in\b', t): return 'dolly_in'
    if re.search(r'\bzoom[- ]?out\b', t): return 'zoom_out'
    if re.search(r'\bzoom[- ]?in\b', t):  return 'zoom_in'
    if re.search(r'low[- ]?angle|high[- ]?angle', t): return 'change_camera_angle'
    if re.search(r'(?:camera|shot|perspective|view)\s+(?:to|into)\s+(?:a\s+)?(?:low|high)', t): return 'change_camera_angle'
    if (bool(re.search(r'transform\s+(?:the\s+)?(?:entire\s+)?(?:video|scene|frame)', t))
            or re.search(r'apply\s+(?:a\s+)?(?:\w+\s+)*(?:art\s+)?style', t)):
        if any(w in t for w in _STYLE_VOCAB): return 'apply_style'
    if any(w in t for w in _STYLE_VOCAB): return 'apply_style'
    if re.search(r'\breplace\b', t):
        if re.search(r'replace\s+(?:the\s+)?(?:(?:\w+\s+){0,6})background', t): return 'replace_background'
        if 'with' in t: return 'replace_object'
    if re.search(r'\bremove\b', t): return 'remove_object'
    if re.search(r'(?:add|apply|enhance).*(?:glow(?:ing)?|aura|flame|lightning|sparkle|decoration\s+effect|stage\s+lighting)', t): return 'add_effect'
    if re.search(r'(?:glowing|neon\s+electric|neon\s+glow|electric\s+glow)', t): return 'add_effect'
    if re.search(r'\b(?:add|insert|place|introduce)\b', t): return 'add_object'
    if re.search(r'(?:change|modify)\s+(?:the\s+)?(?:\w+\s+)*color', t): return 'change_color'
    if re.search(r'color\s+(?:of|to)\s+', t) and re.search(r'\b(?:change|modify|transform)\b', t): return 'change_color'
    return 'edit_motion'

def _extract_new_color(inst):
    t = inst.lower()
    m = re.search(r'\bto\s+(?:a\s+)?(?:(?:vibrant|bright|deep|dark|solid|metallic|neon|bold|glossy){0,2}\s*)'
                  r'([a-z]+(?:\s+[a-z]+){0,2})\s*(?:color|hue|throughout|while|[,.]|$)', t)
    if m:
        val = m.group(1).strip()
        if val not in {'color', 'hue', 'a', 'an', 'the', 'vibrant', 'metallic'}: return val
    return ''

def _extract_color_words(inst):
    t = inst.lower()
    found = []
    for pat in [r'(?:vibrant\s+)?metallic\s+[a-z]+', r'(?:deep|bright|dark|solid)\s+[a-z]+(?:\s+[a-z]+)?',
                r'neon\s+[a-z]+', r'navy\s+blue', r'emerald\s+green']:
        for m in re.finditer(pat, t): found.append(m.group())
    for cw in _COLOR_WORDS:
        if re.search(r'\b' + cw + r'\b', t) and not any(cw in f for f in found):
            found.append(cw)
    return found

def default_target_v15d(action):
    return {
        'replace_background': 'background', 'replace_object': 'object',
        'remove_object': 'object', 'add_object': 'object',
        'increase_amount': 'object', 'change_color': 'subject',
        'apply_style': 'full_frame', 'zoom_out': 'camera_view',
        'zoom_in': 'camera_view', 'dolly_in': 'subject',
        'change_camera_angle': 'subject', 'orbit_camera': 'subject',
        'edit_motion': 'person', 'add_effect': 'subject',
    }.get(action, 'subject')

def default_params_v15d(action, inst):
    t = inst.lower(); p = {}
    if action == 'apply_style':
        for pat, val in [('ukiyo', 'ukiyo-e'), ('ghibli', 'ghibli'), ('watercolor', 'watercolor'),
                         ('oil painting', 'oil_painting'), (r'comic style|american comic|comic book', 'american_comic_style'),
                         (r'\bpixel\b|16-bit', 'pixel_art'), (r'\banime\b', 'anime'), ('cyberpunk', 'cyberpunk')]:
            if re.search(pat, t): p['style'] = val; break
    elif action in ('zoom_in', 'zoom_out', 'dolly_in'):
        p['motion_type'] = action
        if re.search(r'\b(?:slow|smooth|gradual|subtle|steady)\b', t): p['speed'] = 'gradual'
    elif action == 'change_camera_angle':
        if re.search(r'low[- ]?angle', t): p['angle'] = 'low_angle'
        elif re.search(r'high[- ]?angle', t): p['angle'] = 'high_angle'
    elif action == 'orbit_camera': p['trajectory'] = 'arc'
    elif action == 'add_effect': p['effect_type'] = 'glow_or_decoration'
    return p

def extract_target_params_v15d(action, inst):
    """ver16 の extract_primary_target_params と同一実装。
    extraction_actions + change_camera_angle の regex 抽出を含む。"""
    target = default_target_v15d(action)
    params = {}
    t = inst.lower()
    if action == 'change_color':
        m = re.search(r'(?:change|modify)\s+(?:the\s+)?(?:\w+\s+)?color\s+of\s+(?:the\s+)?(.+?)\s+to\s+', inst, re.I)
        if not m: m = re.search(r'(?:change|modify)\s+the\s+(.+?)\s+to\s+', inst, re.I)
        if m: target = m.group(1).strip()
        nc = _extract_new_color(inst); colors = _extract_color_words(inst)
        if nc:
            params['new_color'] = nc
            params['mentioned_colors'] = [nc] + [c for c in colors if c != nc and c not in nc][:5]
    elif action == 'replace_object':
        m = re.search(r'replace\s+(?:the\s+)?(.+?)\s+with\s+', inst, re.I)
        if m: target = m.group(1).strip()
        m2 = re.search(r'\bwith\s+(?:a\s+|an\s+)?(?:\w+\s+){0,3}(.+?)(?:\s+(?:throughout|while)|[,.]|$)', inst, re.I)
        params = {'replacement': {'category': m2.group(1).strip().split()[-1].lower() if m2 else 'replacement', 'viewpoint': 'match_source'}}
    elif action == 'replace_background':
        target = 'background'
        m = re.search(r'\bwith\s+(?:a\s+)?(.+?)(?:\s+featuring|[,.]|$)', inst, re.I)
        params = {'new_scene': {'style': re.sub(r'\s+', '_', m.group(1).strip()[:40].lower())} if m else {}}
    elif action == 'add_object':
        for pat in [r'increase\s+the\s+number\s+of\s+(.+?)\s+(?:in|by|to|on|with)',
                    r'(?:add|insert)\s+(?:a|an|another|additional)\s+(.+?)(?:\s+(?:to|in|on)|[,.]|$)',
                    r'(?:place|introduce)\s+(?:a|an)\s+(.+?)(?:\s+(?:on|in)|[,.]|$)',
                    r'adding\s+(?:a\s+second\s+|more\s+)?(.+?)(?:\s+(?:to|in|on)|[,.]|$)']:
            m = re.search(pat, inst, re.I)
            if m: target = m.group(1).strip(); break
    elif action == 'remove_object':
        m = re.search(r'remove\s+(?:the\s+)?(.+?)(?:\s+(?:from|and)|[,.]|$)', inst, re.I)
        if m: target = m.group(1).strip()
    elif action == 'increase_amount':
        m = re.search(r'increase\s+(?:the\s+)?(?:amount|number)\s+of\s+(.+?)(?:\s+(?:on|in|to)|[,.]|$)', inst, re.I)
        if m: target = m.group(1).strip()
        params = {'density': 'dense'}
    else:
        # ver16 の else ブランチ: camera/style/motion に smart defaults + change_camera_angle 抽出
        params = default_params_v15d(action, inst)
        if action == 'apply_style':   target = 'full_frame'
        elif action in ('zoom_in', 'zoom_out'): target = 'camera_view'
        elif action == 'change_camera_angle':
            m = re.search(r'(?:toward|towards|on|looking\s+(?:up|down)\s+at)\s+(?:the\s+)?([a-z][a-z\s]{1,25}?)(?:\s*[,;.]|$)', t)
            if m: target = m.group(1).strip()
    return target, params

_EXTRACTION_ACTIONS = {'change_color','replace_object','replace_background',
                       'add_object','remove_object','increase_amount'}

def nearest_same_action(inst, action, pool, skip_video):
    best, best_s = None, -1.0
    for c in pool:
        if c['video_path'] == skip_video: continue
        if c['gt_primary']['action'] != action: continue
        s = text_similarity(inst, c['instruction'])
        if s > best_s: best_s, best = s, c
    return best, best_s

def predict_primary_v15d(inst, pool, skip_video):
    """ver16 の predict_primary_task と同一 (near_for_aux は内部のみ)。"""
    action = infer_action(inst)
    target, params = extract_target_params_v15d(action, inst)

    if action not in _EXTRACTION_ACTIONS:
        near, sim = nearest_same_action(inst, action, pool, skip_video)
        if near is not None and sim > 0.15:
            gt = near['gt_primary']
            target = gt.get('target', target)
            merged = copy.deepcopy(gt.get('params', {}))
            for k, v in params.items(): merged[k] = v
            params = merged
        if action == 'apply_style':   target = 'full_frame'
        elif action in ('zoom_in', 'zoom_out'): target = 'camera_view'
        elif action == 'orbit_camera': params['trajectory'] = 'arc'
        elif action == 'edit_motion' and target in ('subject', 'full_frame', 'camera_view', 'face'): target = 'person'
        elif action == 'add_effect': params.setdefault('effect_type', 'glow_or_decoration')

    task = {'action': action, 'target': target, 'constraints': [], 'params': params}
    return task

print('v15d/v16d core ready (ver16 extract_primary_target_params 準拠)')

v15d/v16d core ready (ver16 extract_primary_target_params 準拠)


## 3. v18a — strict scoring で ver17 相当ロジックを再診断
- 意図: `subject` を含む target を 0 点にしたとき、どの action が最も悪化するかを見る。
- 目的: noun 抽出を優先すべき action と、unknown multi-task に `subject` が残るパターンを把握する。
- 判定基準: action 別 target_score と `subject` 含有件数。

In [17]:
def predict_v18a(record):
    task = predict_primary_v15d(record['instruction'], base_records, record['video_path'])
    return {'tasks': [task]}, {'version': 'v18a'}


res_v18a = evaluate_records(base_records, predict_v18a)
print('v18a (ver17 baseline under strict scoring):', res_v18a['overall'])

by_action_tgt = defaultdict(list)
subject_rows = []
for row in res_v18a['rows']:
    key = row['prediction_key']
    rec = next(r for r in base_records if r['prediction_key'] == key)
    act = rec['gt_primary']['action']
    by_action_tgt[act].append(row['target_score'])
    pred_tgt = res_v18a['predictions'][key]['tasks'][0].get('target', '')
    if contains_subject_token(pred_tgt):
        subject_rows.append({
            'action': act,
            'instruction': rec['instruction'],
            'gt_target': rec['gt_primary'].get('target', ''),
            'pred_target': pred_tgt,
        })

print('\n--- action 別 target_score (strict, 昇順) ---')
for act in sorted(by_action_tgt, key=lambda a: sum(by_action_tgt[a]) / len(by_action_tgt[a])):
    sc = by_action_tgt[act]
    print(f'  {act:25s} n={len(sc):2d}  target_avg={sum(sc)/len(sc):.3f}')

print(f'\n合計 target に subject を含む予測: {len(subject_rows)} 件')
print('\n--- subject 含有 target の instruction サンプル (先頭 10件) ---')
for row in subject_rows[:10]:
    print(f'  [{row["action"]:22s}] PRED: {str(row["pred_target"])[:35]:35s} | {row["instruction"][:70]}')

v18a (ver17 baseline under strict scoring): {'action_score': 0.96, 'target_score': 0.68, 'params_score': 0.5329, 'total': 0.7759, 'subject_invalid': 0.04}

--- action 別 target_score (strict, 昇順) ---
  dolly_in                  n= 3  target_avg=0.000
  add_effect                n= 3  target_avg=0.000
  orbit_camera              n= 1  target_avg=0.000
  change_camera_angle       n=10  target_avg=0.100
  add_object                n=11  target_avg=0.273
  zoom_in                   n=11  target_avg=0.545
  change_color              n=11  target_avg=0.818
  edit_motion               n=12  target_avg=0.917
  replace_background        n=12  target_avg=1.000
  apply_style               n=15  target_avg=1.000
  replace_object            n= 6  target_avg=1.000
  remove_object             n= 2  target_avg=1.000
  increase_amount           n= 1  target_avg=1.000
  zoom_out                  n= 2  target_avg=1.000

合計 target に subject を含む予測: 4 件

--- subject 含有 target の instruction サンプル (先頭 10件) ---


## 4. v18b — noun phrase 抽出で `subject` default action を具体化
- 意図: `dolly_in` / `change_camera_angle` / `orbit_camera` / `add_effect` など、`subject` に落ちやすい action で instruction から具体 noun phrase を拾う。
- 目的: retrieval なしでも `subject` を具体 target に置き換えられる範囲を広げる。
- 判定基準: strict target_score と `subject` 含有件数が v18a より改善するか。

In [18]:
_GENERIC_TARGETS = {
    'subject', 'person', 'object', 'full frame', 'camera view', 'background',
    'foreground', 'video', 'scene', 'frame', 'camera', 'angle',
}

_HEAD_HINTS = {
    'face', 'profile', 'man', 'woman', 'men', 'car', 'speaker', 'presenter',
    'player', 'guitar', 'strings', 'grinder', 'bed', 'background', 'microphone',
    'panda', 'planet', 'jam', 'necktie', 'shirt', 'crowd', 'hands', 'hair'
}


def clean_candidate_phrase(text):
    if not text:
        return ''
    text = str(text).strip().lower()
    text = re.split(r'\b(?:while|with|without|throughout|starting|ending|keeping|keep|maintain(?:ing)?|preserve|ensure|guarantee|make\s+sure|that|which|who|by|so\s+that)\b', text, maxsplit=1)[0]
    text = re.sub(r'^(?:the|a|an)\s+', '', text)
    text = re.sub(r'[^a-z0-9\s\'\-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip(' .,;:')
    if not text:
        return ''
    words = text.split()
    if len(words) > 6:
        text = ' '.join(words[:6])
    if normalize_text(text) in _GENERIC_TARGETS:
        return ''
    if re.search(r'\b(?:video|scene|frame|camera|angle)\b', normalize_text(text)):
        return ''
    return text


def rank_candidate(text):
    nt = normalize_text(text)
    words = nt.split()
    score = len(words)
    if any(h in words for h in _HEAD_HINTS):
        score += 3
    if "'s" in text:
        score += 1
    if len(words) >= 2:
        score += 1
    return score


def choose_best_candidate(candidates):
    cleaned = []
    seen = set()
    for cand in candidates:
        cc = clean_candidate_phrase(cand)
        if not cc:
            continue
        key = normalize_text(cc)
        if key in seen:
            continue
        seen.add(key)
        cleaned.append(cc)
    if not cleaned:
        return ''
    cleaned.sort(key=lambda c: (rank_candidate(c), len(c)), reverse=True)
    return cleaned[0]


def rewrite_subject_text(text, concrete_target=''):
    if text is None:
        return text
    out = str(text)
    out = re.sub(r"\bsubject's\b\s*", '', out, flags=re.I)
    if concrete_target:
        out = re.sub(r'\bsubject\b', concrete_target, out, flags=re.I)
    else:
        out = re.sub(r'\bsubject\b', '', out, flags=re.I)
    out = re.sub(r'\s+', ' ', out).strip(' .,;:')
    return out


def rewrite_subject_target(target, concrete_target=''):
    if isinstance(target, list):
        return [rewrite_subject_text(v, concrete_target) for v in target]
    return rewrite_subject_text(target, concrete_target)


def is_generic_prediction_target(target):
    nt = target_text(target)
    return contains_subject_token(target) or nt in {'person', 'object', ''}


def extract_target_base_v18(action, inst):
    t = inst.lower()
    target, params = extract_target_params_v15d(action, inst)
    if action in _EXTRACTION_ACTIONS:
        return target, params

    params = default_params_v15d(action, inst)
    if action == 'apply_style':
        return 'full_frame', params
    if action in ('zoom_in', 'zoom_out'):
        return 'camera_view', params

    if action == 'dolly_in':
        m = re.search(r'dolly[- ]?in\s+(?:to\s+|toward\s+|towards\s+)?(?:the\s+)?([a-z][a-z\s\'\-]{1,35}?)(?:\s*[,;.]|\s+(?:in|on|at|to|while|from)|$)', t)
        if m:
            return m.group(1).strip(), params
        return 'subject', params

    if action == 'change_camera_angle':
        m = re.search(r'(?:toward|towards|at|on|facing|looking\s+(?:up|down)\s+at)\s+(?:the\s+)?([a-z][a-z\s\'\-]{1,35}?)(?:\s*[,;.]|\s+(?:in|on|at|to|with|while|from|that|and)|$)', t)
        if m:
            return m.group(1).strip(), params
        return target, params

    if action == 'orbit_camera':
        m = re.search(r'(?:orbit|arc|revolv\w+|circl\w+)\s+(?:around\s+)?(?:the\s+)?([a-z][a-z\s\'\-]{1,35}?)(?:\s*[,;.]|\s+(?:in|on|at|to|with|while|from|that|and)|$)', t)
        if m:
            return m.group(1).strip(), params
        return 'subject', params

    if action == 'add_effect':
        m = re.search(r'(?:add|apply|enhance)\s+(?:\w+\s+){0,6}?(?:to|on|around)\s+(?:the\s+)?([a-z][a-z\s\'\-]{1,35}?)(?:\s*[,;.]|\s+(?:in|on|at|to|with|while|from|that|and)|$)', t)
        if m:
            return m.group(1).strip(), params
        return 'subject', params

    if action == 'edit_motion':
        m = re.search(r'(?:make|have|get|let)\s+the\s+([a-z][a-z\s\']{1,35}?)\s+(?:wave|nod|spin|jump|walk|run|dance|move|perform|do|look|turn|bow)', t)
        if m:
            return m.group(1).strip(), params
        return 'person', params

    return default_target_v15d(action), params


def extract_concrete_target_v18(action, inst):
    t = inst.lower()
    candidates = []
    pattern_groups = {
        'dolly_in': [
            r'(?:toward|towards|to)\s+(?:the\s+)?([a-z][a-z\s\'\-]{1,40}?)(?:\s*[,;.]|\s+(?:starting|ending|while|with|keeping|keep|maintain|preserve|ensure)|$)',
            r'(?:closer\s+on|focus(?:ing)?\s+on)\s+(?:the\s+)?([a-z][a-z\s\'\-]{1,40}?)(?:\s*[,;.]|\s+(?:while|with|keep|maintain|preserve|ensure)|$)',
        ],
        'change_camera_angle': [
            r'look(?:ing)?\s+(?:up|down)\s+at\s+(?:the\s+)?([a-z][a-z\s\'\-]{1,40}?)(?:\s*[,;.]|\s+(?:to|while|with|keep|maintain|preserve|ensure)|$)',
            r'making\s+the\s+camera\s+look\s+(?:up|down)\s+at\s+(?:the\s+)?([a-z][a-z\s\'\-]{1,40}?)(?:\s*[,;.]|\s+(?:to|while|with|keep|maintain|preserve|ensure)|$)',
            r'(?:low|high)\s+angle\s+(?:shot\s*,?\s*)?(?:looking\s+(?:up|down)\s+at\s+)?(?:the\s+)?([a-z][a-z\s\'\-]{1,40}?)(?:\s*[,;.]|\s+(?:from|to|while|with|keep|maintain|preserve|ensure)|$)',
        ],
        'orbit_camera': [
            r'(?:around|revolving\s+around|revolve\s+around|orbit(?:ing)?\s+around)\s+(?:the\s+)?([a-z][a-z\s\'\-]{1,40}?)(?:\s*[,;.]|\s+(?:to|while|with|keep|maintain|preserve|ensure)|$)',
        ],
        'add_effect': [
            r'enhance\s+(?:the\s+)?([a-z][a-z\s\'\-]{1,40}?)\s+by\s+adding',
            r'(?:apply|add)\s+.{0,40}?(?:to|on|around)\s+(?:the\s+)?([a-z][a-z\s\'\-]{1,40}?)(?:\s*[,;.]|\s+(?:to|while|with|keep|maintain|preserve|ensure)|$)',
            r'outlin(?:e|es)\s+(?:his|her|their|the)\s+([a-z][a-z\s\'\-]{1,40}?)(?:\s*[,;.]|\s+(?:throughout|while|with|keep|maintain|preserve|ensure)|$)',
        ],
        'edit_motion': [
            r'(?:make|have|get|let)\s+the\s+([a-z][a-z\s\'\-]{1,40}?)\s+(?:wave|nod|spin|jump|walk|run|dance|move|perform|do|look|turn|bow)',
            r'([a-z][a-z\s\'\-]{1,30}?)\'s\s+(?:hand|arm|head|body|leg|motion|gesture|movement)',
        ],
    }
    generic_patterns = [
        r'(?:the|a|an)\s+([a-z][a-z\s\'\-]{1,30}?(?:face|profile|speaker|presenter|player|man|woman|men|car|guitar|strings|grinder|bed|microphone|panda|planet|jam|necktie|shirt|hair))\b',
        r'([a-z][a-z\s\'\-]{1,30}?)\s+(?:face|profile)\b',
    ]
    for pat in pattern_groups.get(action, []):
        for m in re.finditer(pat, t):
            candidates.append(m.group(1))
    for pat in generic_patterns:
        for m in re.finditer(pat, t):
            candidates.append(m.group(1))
    return choose_best_candidate(candidates)


def predict_v18b(record):
    inst = record['instruction']
    action = infer_action(inst)
    target, params = extract_target_base_v18(action, inst)
    noun_target = extract_concrete_target_v18(action, inst)
    if contains_subject_token(target):
        target = rewrite_subject_target(target, noun_target)
    if is_generic_prediction_target(target) and noun_target:
        target = noun_target
    task = {'action': action, 'target': target, 'constraints': [], 'params': params}
    return {'tasks': [task]}, {'version': 'v18b', 'action': action, 'target': target, 'noun_target': noun_target}


res_v18b = evaluate_records(base_records, predict_v18b)
print('v18b (noun phrase extraction):', res_v18b['overall'])
print('\nv18a -> v18b delta:')
for k in ['action_score', 'target_score', 'params_score', 'total']:
    delta = res_v18b['overall'][k] - res_v18a['overall'][k]
    print(f'  {k:15s}: {res_v18a["overall"][k]:.4f} -> {res_v18b["overall"][k]:.4f}  ({delta:+.4f})')

v18b (noun phrase extraction): {'action_score': 1.0, 'target_score': 0.66, 'params_score': 0.5489, 'total': 0.7967, 'subject_invalid': 0.0}

v18a -> v18b delta:
  action_score   : 0.9600 -> 1.0000  (+0.0400)
  target_score   : 0.6800 -> 0.6600  (-0.0200)
  params_score   : 0.5329 -> 0.5489  (+0.0160)
  total          : 0.7759 -> 0.7967  (+0.0208)


### v18b action 別詳細
- strict target_score の改善幅と、`subject` がどれだけ消えたかを確認する。

In [19]:
def per_action_scores(res, records):
    ba = defaultdict(lambda: {'total': [], 'target': [], 'params': []})
    for row in res['rows']:
        key = row['prediction_key']
        rec = next(r for r in records if r['prediction_key'] == key)
        act = rec['gt_primary']['action']
        ba[act]['total'].append(row['total'])
        ba[act]['target'].append(row['target_score'])
        ba[act]['params'].append(row['params_score'])
    return {act: {k: round(sum(v) / len(v), 4) for k, v in d.items()} for act, d in ba.items()}


sc_a = per_action_scores(res_v18a, base_records)
sc_b = per_action_scores(res_v18b, base_records)

print(f'{"action":25s}  {"total_a":>8s}  {"total_b":>8s}  {"delta":>8s}  {"tgt_a":>6s}  {"tgt_b":>6s}')
for act in sorted(sc_a):
    a, b = sc_a[act], sc_b[act]
    print(f'{act:25s}  {a["total"]:8.4f}  {b["total"]:8.4f}  {b["total"] - a["total"]:+8.4f}  {a["target"]:6.3f}  {b["target"]:6.3f}')

subject_remain = [
    (rec['gt_primary']['action'], rec['instruction'][:70], res_v18b['predictions'][rec['prediction_key']]['tasks'][0].get('target', ''))
    for rec in base_records
    if contains_subject_token(res_v18b['predictions'][rec['prediction_key']]['tasks'][0].get('target', ''))
]
print(f'\nv18b 後も subject を含む予測: {len(subject_remain)} 件')
for act, inst, pred in subject_remain[:8]:
    print(f'  [{act:22s}] PRED={str(pred)[:30]:30s} | {inst}')

action                      total_a   total_b     delta   tgt_a   tgt_b
add_effect                   0.7500    0.7500   +0.0000   0.000   0.000
add_object                   0.5545    0.5545   +0.0000   0.273   0.273
apply_style                  1.0000    1.0000   +0.0000   1.000   1.000
change_camera_angle          0.7400    0.9200   +0.1800   0.100   0.600
change_color                 0.7726    0.8526   +0.0800   0.818   0.909
dolly_in                     0.5333    0.7333   +0.2000   0.000   0.000
edit_motion                  0.7417    0.5583   -0.1834   0.917   0.167
increase_amount              0.7600    0.7600   +0.0000   1.000   1.000
orbit_camera                 0.0000    1.0000   +1.0000   0.000   1.000
remove_object                1.0000    1.0000   +0.0000   1.000   1.000
replace_background           0.7133    0.7133   +0.0000   1.000   1.000
replace_object               0.7950    0.7950   +0.0000   1.000   1.000
zoom_in                      0.8955    0.8955   +0.0000   0.545 

## 5. v18c — v18b + retrieval fallback + subject phrase rewrite
- 意図: noun 抽出だけで target が決まらない場合、近傍 GT target で補完する。
- 追加方針: `subject's ...` のような phrase も downstream 向けに書き換える。
- 目的: strict scoring を保ったまま、single-task の best を更新する。
- 判定基準: v18b より target_score / total が改善するか。

In [20]:
_V18C_RETRIEVAL_PRIMARY = {'edit_motion', 'dolly_in'}


def predict_v18c(record):
    inst = record['instruction']
    action = infer_action(inst)
    target, params = extract_target_base_v18(action, inst)
    noun_target = extract_concrete_target_v18(action, inst)
    near, sim = nearest_same_action(inst, action, base_records, record['video_path'])

    if action in _V18C_RETRIEVAL_PRIMARY and near is not None and sim > 0.12:
        retrieved_target = near['gt_primary'].get('target', target)
        if contains_subject_token(retrieved_target):
            retrieved_target = rewrite_subject_target(retrieved_target, noun_target)
        if not contains_subject_token(retrieved_target) and normalize_text(retrieved_target) not in {'', 'subject', 'person', 'object'}:
            target = retrieved_target
        elif noun_target:
            target = noun_target
    else:
        if contains_subject_token(target):
            target = rewrite_subject_target(target, noun_target)
        if is_generic_prediction_target(target) and noun_target:
            target = noun_target
        if is_generic_prediction_target(target) and near is not None and sim > 0.12:
            retrieved_target = near['gt_primary'].get('target', target)
            if contains_subject_token(retrieved_target):
                retrieved_target = rewrite_subject_target(retrieved_target, noun_target)
            if not contains_subject_token(retrieved_target) and normalize_text(retrieved_target) not in {'', 'subject', 'person', 'object'}:
                target = retrieved_target

    if action not in _EXTRACTION_ACTIONS and near is not None and sim > 0.15:
        merged = copy.deepcopy(near['gt_primary'].get('params', {}))
        for k, v in params.items():
            merged[k] = v
        params = merged
        if action == 'apply_style':
            target = 'full_frame'
        elif action in ('zoom_in', 'zoom_out'):
            target = 'camera_view'
        elif action == 'orbit_camera':
            params['trajectory'] = 'arc'
        elif action == 'add_effect':
            params.setdefault('effect_type', 'glow_or_decoration')

    task = {'action': action, 'target': target, 'constraints': [], 'params': params}
    return {'tasks': [task]}, {'version': 'v18c', 'action': action, 'target': target, 'noun_target': noun_target}


res_v18c = evaluate_records(base_records, predict_v18c)
print('v18c (noun extraction + selective retrieval):', res_v18c['overall'])

print('\n--- v18a/b/c 比較 ---')
for name, res in [('v18a', res_v18a), ('v18b', res_v18b), ('v18c', res_v18c)]:
    print(f'  {name}: {res["overall"]}')

v18c (noun extraction + selective retrieval): {'action_score': 1.0, 'target_score': 0.66, 'params_score': 0.5623, 'total': 0.8007, 'subject_invalid': 0.0}

--- v18a/b/c 比較 ---
  v18a: {'action_score': 0.96, 'target_score': 0.68, 'params_score': 0.5329, 'total': 0.7759, 'subject_invalid': 0.04}
  v18b: {'action_score': 1.0, 'target_score': 0.66, 'params_score': 0.5489, 'total': 0.7967, 'subject_invalid': 0.0}
  v18c: {'action_score': 1.0, 'target_score': 0.66, 'params_score': 0.5623, 'total': 0.8007, 'subject_invalid': 0.0}


## 6. v18d — v18c primary を使う strict multi-task 版
- 方針:
  - primary task は v18c の concrete target を使う。
  - aux task は ver16d の action-gated retrieval を維持する。
  - ただし task 内 target に `subject` が残る場合は、primary target を使って書き換える。
- 目的: strict multi-task 評価でも `subject` をできるだけ残さず、downstream で使いやすい task 列にする。

In [21]:
# v16d の aux 設定を再利用
_AUX_MAP = {
    'add_effect': ['match_effect_lighting', 'stabilize_effect'],
    'add_object': ['match_appearance', 'blend_instances', 'stabilize_instances'],
    'apply_style': ['enhance_style_details', 'stabilize_style', 'preserve_layout'],
    'change_camera_angle': ['adjust_perspective', 'refine_mask'],
    'change_color': ['preserve_material_appearance', 'preserve_objects', 'refine_mask'],
    'dolly_in': ['preserve_framing'],
    'edit_motion': ['stabilize_motion', 'refine_mask', 'preserve_objects', 'preserve_identity'],
    'increase_amount': ['match_appearance', 'stabilize_instances'],
    'orbit_camera': ['stabilize_motion'],
    'remove_object': ['inpaint_background', 'stabilize_inpaint'],
    'replace_background': ['match_lighting', 'refine_mask', 'stabilize_composite', 'match_background_camera_properties', 'preserve_foreground'],
    'replace_object': ['align_replacement', 'match_appearance', 'refine_mask'],
    'zoom_in': ['stabilize_motion', 'preserve_focus'],
    'zoom_out': ['preserve_focus', 'stabilize_motion'],
}
_AUX_TARGET_OVERRIDE = {
    'enhance_style_details': 'full_frame',
    'stabilize_style': 'full_frame',
    'stabilize_composite': 'full_frame',
    'preserve_layout': 'scene_layout',
    'match_background_camera_properties': 'background',
    'preserve_foreground': 'foreground',
    'inpaint_background': 'background',
}


def make_aux_task(aux_action, primary_target):
    tgt = _AUX_TARGET_OVERRIDE.get(aux_action, primary_target)
    return {'action': aux_action, 'target': tgt, 'constraints': [], 'params': {}}


_V16D_HELP_B = {'change_camera_angle', 'edit_motion', 'zoom_in', 'apply_style', 'add_effect', 'remove_object'}
_V16D_HELP_C = {'increase_amount', 'orbit_camera'}
_V16D_TH_MAP = {
    'change_camera_angle': 0.12,
    'edit_motion': 0.28,
    'zoom_in': 0.12,
    'apply_style': 0.28,
    'add_effect': 0.12,
    'remove_object': 0.12,
}


def sanitize_task_target(task, concrete_target):
    task = copy.deepcopy(task)
    tgt = task.get('target', '')
    if contains_subject_token(tgt):
        task['target'] = rewrite_subject_target(tgt, concrete_target)
    return task


def predict_v18d(record):
    inst = record['instruction']
    primary = predict_v18c(record)[0]['tasks'][0]
    action = primary['action']
    target = primary['target']

    near, sim = nearest_same_action(inst, action, base_records, record['video_path'])
    aux_actions = _AUX_MAP.get(action, [])
    tasks = [primary] + [make_aux_task(a, target) for a in aux_actions]

    if action in _V16D_HELP_C:
        n_aux = max(0, len(near['gt_tasks']) - 1) if near is not None else 0
        tasks = [primary] + [make_aux_task(a, target) for a in aux_actions[:n_aux]]

    th = _V16D_TH_MAP.get(action, 1.0)
    if near is not None and action in _V16D_HELP_B and sim >= th:
        tasks = [primary] + copy.deepcopy(near['gt_tasks'][1:])

    tasks = [sanitize_task_target(t, target) for t in tasks]
    return {'tasks': tasks}


res_v18d = evaluate_multi(base_records, predict_v18d)
print('v18d (strict multi-task with concrete targets):', res_v18d['overall'])

v18d (strict multi-task with concrete targets): {'coverage': 0.8071, 'precision': 0.7569, 'count_alignment': 0.7565, 'total': 0.7845, 'subject_invalid': 0.0}


In [22]:
# ---- v16d 参照実装の再現チェック ----
# predict_primary_v15d を evaluate_multi で評価して、v16a(=0.7374) と一致するか確認

def predict_v16_ref(record):
    """v16d の predict_primary_task を完全再現 (near_for_aux を返す版)。"""
    inst = record['instruction']
    action = infer_action(inst)
    target, params = extract_target_params_v15d(action, inst)

    if action not in _EXTRACTION_ACTIONS:
        near, sim = nearest_same_action(inst, action, base_records, record['video_path'])
        if near is not None and sim > 0.15:
            gt = near['gt_primary']
            target = gt.get('target', target)
            merged = copy.deepcopy(gt.get('params', {}))
            for k, v in params.items(): merged[k] = v
            params = merged
        if action == 'apply_style':   target = 'full_frame'
        elif action in ('zoom_in', 'zoom_out'): target = 'camera_view'
        elif action == 'orbit_camera': params['trajectory'] = 'arc'
        elif action == 'edit_motion' and target in ('subject', 'full_frame', 'camera_view', 'face'): target = 'person'
        elif action == 'add_effect': params.setdefault('effect_type', 'glow_or_decoration')
        near_for_aux, sim_for_aux = near, (sim if near is not None else 0.0)
    else:
        near_for_aux, sim_for_aux = nearest_same_action(inst, action, base_records, record['video_path'])

    primary = {'action': action, 'target': target, 'constraints': [], 'params': params}

    # v16d aux gating
    aux_actions = _AUX_MAP.get(action, [])
    tasks = [primary] + [make_aux_task(a, target) for a in aux_actions]
    if action in _V16D_HELP_C:
        n_aux = max(0, len(near_for_aux['gt_tasks']) - 1) if near_for_aux is not None else 0
        tasks = [primary] + [make_aux_task(a, target) for a in aux_actions[:n_aux]]
    th = _V16D_TH_MAP.get(action, 1.0)
    if near_for_aux is not None and action in _V16D_HELP_B and sim_for_aux >= th:
        tasks = [primary] + copy.deepcopy(near_for_aux['gt_tasks'][1:])

    return {'tasks': tasks}

res_v16ref = evaluate_multi(base_records, predict_v16_ref)
print('v16_ref (v16d exact reimplementation):', res_v16ref['overall'])
print('v16d ref target:                        coverage=0.8125, precision=0.7630, count_alignment=0.7565, total=0.7896')

v16_ref (v16d exact reimplementation): {'coverage': 0.7543, 'precision': 0.709, 'count_alignment': 0.699, 'total': 0.7329, 'subject_invalid': 0.07}
v16d ref target:                        coverage=0.8125, precision=0.7630, count_alignment=0.7565, total=0.7896


## 7. 全モデル比較 + strict multi-task 診断
- v18 系の known benchmark を並べて、single-task / multi-task の best を確認する。
- strict scoring なので、`subject` を残す設計は ver17 より不利になる。

In [23]:
print('=== single-task known benchmark (ver18 strict) ===')
for name, res in [('v18a (strict baseline)', res_v18a), ('v18b (noun extraction)', res_v18b), ('v18c (noun+retrieval)', res_v18c)]:
    print(f'  {name:35s}  {res["overall"]}')

print()
print('=== multi-task known benchmark (strict) ===')
print(f'  v16d strict ref        {res_v16ref["overall"]}')
print(f'  v18d                   {res_v18d["overall"]}')

by_act = defaultdict(lambda: {'total': [], 'cov': [], 'prec': []})
for row in res_v18d['rows']:
    rec = next(r for r in base_records if r['prediction_key'] == row['prediction_key'])
    act = rec['gt_primary']['action']
    by_act[act]['total'].append(row['total'])
    by_act[act]['cov'].append(row['coverage'])
    by_act[act]['prec'].append(row['precision'])

print('\n--- v18d action 別 multi-task total ---')
for act in sorted(by_act, key=lambda a: sum(by_act[a]['total']) / len(by_act[a]['total'])):
    d = by_act[act]
    n = len(d['total'])
    print(f'  {act:25s} n={n:2d}  total={sum(d["total"])/n:.3f}  cov={sum(d["cov"])/n:.3f}  prec={sum(d["prec"])/n:.3f}')

subject_multi = []
for row in res_v18d['rows']:
    rec = next(r for r in base_records if r['prediction_key'] == row['prediction_key'])
    pred_tasks = res_v18d['predictions'][rec['prediction_key']]['tasks']
    if any(contains_subject_token(t.get('target', '')) for t in pred_tasks):
        subject_multi.append((rec['gt_primary']['action'], rec['instruction'][:80], pred_tasks[0].get('target', '')))

print(f'\n--- v18d で subject を含む task が残る record: {len(subject_multi)} 件 ---')
for act, inst, tgt in subject_multi[:10]:
    print(f'  [{act:22s}] PRED={str(tgt)[:30]:30s} | {inst}')

=== single-task known benchmark (ver18 strict) ===
  v18a (strict baseline)               {'action_score': 0.96, 'target_score': 0.68, 'params_score': 0.5329, 'total': 0.7759, 'subject_invalid': 0.04}
  v18b (noun extraction)               {'action_score': 1.0, 'target_score': 0.66, 'params_score': 0.5489, 'total': 0.7967, 'subject_invalid': 0.0}
  v18c (noun+retrieval)                {'action_score': 1.0, 'target_score': 0.66, 'params_score': 0.5623, 'total': 0.8007, 'subject_invalid': 0.0}

=== multi-task known benchmark (strict) ===
  v16d strict ref        {'coverage': 0.7543, 'precision': 0.709, 'count_alignment': 0.699, 'total': 0.7329, 'subject_invalid': 0.07}
  v18d                   {'coverage': 0.8071, 'precision': 0.7569, 'count_alignment': 0.7565, 'total': 0.7845, 'subject_invalid': 0.0}

--- v18d action 別 multi-task total ---
  dolly_in                  n= 3  total=0.588  cov=0.608  prec=0.533
  edit_motion               n=12  total=0.661  cov=0.659  prec=0.626
  add_objec

## 8. unknown 評価 + 保存
- single-task: v18a / v18b / v18c を strict scoring で評価し、best を保存する。
- multi-task: v18d を評価・保存する。
- 出力先は `notebook/ver18_outputs/`。

In [24]:
unknown_single = {
    'v18a': evaluate_records(unknown_records, predict_v18a),
    'v18b': evaluate_records(unknown_records, predict_v18b),
    'v18c': evaluate_records(unknown_records, predict_v18c),
}
print('=== single-task unknown benchmark (ver18 strict) ===')
for name, res in sorted(unknown_single.items(), key=lambda x: x[1]['overall']['total'], reverse=True):
    print(f'  {name:6s}  {res["overall"]}')

res_v18d_unknown = evaluate_multi(unknown_records, predict_v18d)
print()
print('=== multi-task unknown benchmark (strict) ===')
print(f'  v16d strict ref  {evaluate_multi(unknown_records, predict_v16_ref)["overall"]}')
print(f'  v18d             {res_v18d_unknown["overall"]}')

best_single_name = max(unknown_single, key=lambda x: unknown_single[x]['overall']['total'])
best_single_res = unknown_single[best_single_name]
scbk = {r['prediction_key']: r for r in best_single_res['rows']}
rows_single = []
for rec in unknown_records:
    key = rec['prediction_key']
    rows_single.append({
        'prediction_key': key,
        'video_path': rec['video_path'],
        'variant': rec.get('variant', 'base'),
        'instruction': rec['instruction'],
        'gt_primary': rec['gt_primary'],
        'prediction': best_single_res['predictions'][key],
        'scores': scbk[key],
    })
(OUTPUT_DIR / f'unknown_predictions_{best_single_name}_single.json').write_text(
    json.dumps(rows_single, ensure_ascii=False, indent=2), encoding='utf-8'
)

scbk_multi = {r['prediction_key']: r for r in res_v18d_unknown['rows']}
rows_multi = []
for rec in unknown_records:
    key = rec['prediction_key']
    rows_multi.append({
        'prediction_key': key,
        'video_path': rec['video_path'],
        'variant': rec.get('variant', 'base'),
        'instruction': rec['instruction'],
        'gt_tasks': rec['gt_tasks'],
        'prediction': res_v18d_unknown['predictions'][key],
        'scores': scbk_multi[key],
    })
(OUTPUT_DIR / 'unknown_predictions_v18d_multi.json').write_text(
    json.dumps(rows_multi, ensure_ascii=False, indent=2), encoding='utf-8'
)

analysis = {
    'single_task': {
        'known': {k: v['overall'] for k, v in [('v18a', res_v18a), ('v18b', res_v18b), ('v18c', res_v18c)]},
        'unknown': {k: v['overall'] for k, v in unknown_single.items()},
        'best_unknown': best_single_name,
    },
    'multi_task': {
        'known_v16_strict_ref': res_v16ref['overall'],
        'known_v18d': res_v18d['overall'],
        'unknown_v18d': res_v18d_unknown['overall'],
    },
}
(OUTPUT_DIR / 'analysis_ver18.json').write_text(
    json.dumps(analysis, ensure_ascii=False, indent=2), encoding='utf-8'
)
print(f'\nsaved best_single={best_single_name} to {OUTPUT_DIR}')

=== single-task unknown benchmark (ver18 strict) ===
  v18c    {'action_score': 0.9667, 'target_score': 0.6183, 'params_score': 0.5612, 'total': 0.7754, 'subject_invalid': 0.0}
  v18b    {'action_score': 0.9667, 'target_score': 0.6183, 'params_score': 0.5446, 'total': 0.7704, 'subject_invalid': 0.0017}
  v18a    {'action_score': 0.9267, 'target_score': 0.6483, 'params_score': 0.5319, 'total': 0.7526, 'subject_invalid': 0.0433}

=== multi-task unknown benchmark (strict) ===
  v16d strict ref  {'coverage': 0.7329, 'precision': 0.6843, 'count_alignment': 0.6827, 'total': 0.7109, 'subject_invalid': 0.0683}
  v18d             {'coverage': 0.7824, 'precision': 0.7275, 'count_alignment': 0.7381, 'total': 0.7587, 'subject_invalid': 0.0}

saved best_single=v18c to /workspace/notebook/ver18_outputs


## 9. エラー分析 — worst 10 records (v18c single-task unknown)
- strict scoring 下で残る誤りを確認し、次の改善仮説を立てる。 #

In [25]:
worst = sorted(unknown_single['v18c']['rows'], key=lambda x: x['total'])[:10]
print('worst 10: v18c unknown')
for row in worst:
    key = row['prediction_key']
    rec = next(r for r in unknown_records if r['prediction_key'] == key)
    pred = unknown_single['v18c']['predictions'][key]['tasks'][0]
    print('=' * 90)
    print(f'  action: GT={rec["gt_primary"]["action"]:22s}  PRED={pred["action"]:22s}  act={row["action_score"]}')
    print(f'  target: GT={str(rec["gt_primary"].get("target", ""))[:35]:35s}  PRED={str(pred.get("target", ""))[:35]:35s}  tgt={row["target_score"]}')
    print(f'  params_score={row["params_score"]}  total={row["total"]}')
    print(f'  subject_in_pred={contains_subject_token(pred.get("target", ""))}')
    print(f'  inst: {rec["instruction"][:120]}')

worst 10: v18c unknown
  action: GT=replace_background      PRED=edit_motion             act=0.0
  target: GT=background_behind_speaker            PRED=person                               tgt=0.0
  params_score=0.0  total=0.0
  subject_in_pred=False
  inst: Swap the entire outdoor background behind the speaker with a sleek, modern automotive showroom featuring soft studio lig
  action: GT=replace_object          PRED=edit_motion             act=0.0
  target: GT=woman's black knit beanie            PRED=hat meets her blonde hair            tgt=0.0
  params_score=0.0  total=0.0
  subject_in_pred=False
  inst: Swap the woman's black knit beanie with a bright red wool hat throughout the entire video sequence. Make sure the new ha
  action: GT=replace_background      PRED=edit_motion             act=0.0
  target: GT=black_background                     PRED=focus on the presenter               tgt=0.0
  params_score=0.0  total=0.0
  subject_in_pred=False
  inst: Swap the entire solid black

## 修正済み GT (ver10) での ver17 再検証

**背景**: GT の `target` から `subject` プレースホルダーを削除（44 箇所置換）
- 修正前 GT: `annotations_gt_task_ver09.json`
- 修正後 GT: `annotations_gt_task_ver10.json`

**目的**: 修正後 GT での ver17 予測の再評価 → スコア変化を計測

**方法**:
1. ver17 の予測ファイル（single + multi）を読み込み
2. 修正前 GT（ver09）での評価結果を比較対象として読み込み
3. 修正後 GT（ver10）で再評価する
4. 差分を表示

In [27]:
# ---- Load ver17 predictions ----
VER17_DIR = NOTEBOOK_DIR / 'ver17_outputs'
v17_single_path = VER17_DIR / 'unknown_predictions_v17c_single.json'
v17_multi_path = VER17_DIR / 'unknown_predictions_v17d_multi.json'
v17_analysis_path = VER17_DIR / 'analysis_ver17.json'

v17_single = json.loads(v17_single_path.read_text(encoding='utf-8'))
v17_multi = json.loads(v17_multi_path.read_text(encoding='utf-8'))
v17_analysis_old = json.loads(v17_analysis_path.read_text(encoding='utf-8'))

print(f'loaded v17_single: {len(v17_single)}')
print(f'loaded v17_multi: {len(v17_multi)}')
print(f'loaded v17_analysis_old (GT_ver09 ベース): {list(v17_analysis_old.keys())}')


loaded v17_single: 600
loaded v17_multi: 600
loaded v17_analysis_old (GT_ver09 ベース): ['single_task', 'multi_task']


In [36]:
# ---- Debug: Check base_records and v17 predictions structure ----
print('\n=== DEBUG: base_records Structure ===')
if base_records:
    sample_rec = base_records[0]
    print(f'Sample base_record keys: {sample_rec.keys()}')
    print(f'prediction_key: {sample_rec.get("prediction_key")}')
    print(f'tasks count: {len(sample_rec.get("tasks", []))}')
    if sample_rec.get('tasks'):
        print(f'first task keys: {sample_rec["tasks"][0].keys()}')

print('\n=== DEBUG: v17_single Structure ===')
if v17_single:
    sample_v17 = v17_single[0]
    print(f'Sample v17 keys: {sample_v17.keys()}')
    print(f'prediction_key: {sample_v17.get("prediction_key")}')
    print(f'prediction.tasks count: {len(sample_v17.get("prediction", {}).get("tasks", []))}')
    if sample_v17.get('prediction', {}).get('tasks'):
        print(f'first pred task keys: {sample_v17["prediction"]["tasks"][0].keys()}')

# Check if keys match
if base_records and v17_single:
    br_key = base_records[0].get('prediction_key')
    v17_key = v17_single[0].get('prediction_key')
    print(f'\nKey match check:')
    print(f'  base_records[0].prediction_key: {br_key}')
    print(f'  v17_single[0].prediction_key: {v17_key}')
    print(f'  Match: {br_key == v17_key}')
    
    # Check if v17_key exists in any base_records
    matching_count = sum(1 for r in base_records if r.get('prediction_key') == v17_key)
    print(f'  Matches in base_records: {matching_count}')



=== DEBUG: base_records Structure ===
Sample base_record keys: dict_keys(['video_path', 'prediction_key', 'class', 'subclass', 'instruction', 'gt_tasks', 'gt_primary'])
prediction_key: wyzi9GNZFMU_0_0to121.mp4::base
tasks count: 0

=== DEBUG: v17_single Structure ===
Sample v17 keys: dict_keys(['prediction_key', 'video_path', 'variant', 'instruction', 'gt_primary', 'prediction', 'scores'])
prediction_key: wyzi9GNZFMU_0_0to121.mp4::annotations_grouped_ver01:ver2
prediction.tasks count: 1
first pred task keys: dict_keys(['action', 'target', 'constraints', 'params'])

Key match check:
  base_records[0].prediction_key: wyzi9GNZFMU_0_0to121.mp4::base
  v17_single[0].prediction_key: wyzi9GNZFMU_0_0to121.mp4::annotations_grouped_ver01:ver2
  Match: False
  Matches in base_records: 0


In [37]:
# ---- Debug: Check v17_analysis_old structure ----
print('\n=== DEBUG: v17_analysis_old Structure ===')
print(f'Keys: {v17_analysis_old.keys()}')
for key in v17_analysis_old.keys():
    print(f'\n{key}:')
    val = v17_analysis_old[key]
    if isinstance(val, dict):
        print(f'  Type: dict, Keys: {val.keys()}')
        # Print first few values
        for k in list(val.keys())[:5]:
            print(f'    {k}: {val[k]}')



=== DEBUG: v17_analysis_old Structure ===
Keys: dict_keys(['single_task', 'multi_task'])

single_task:
  Type: dict, Keys: dict_keys(['known', 'unknown', 'best_unknown'])
    known: {'v17a': {'action_score': 1.0, 'target_score': 0.69, 'params_score': 0.5623, 'total': 0.8067}, 'v17b': {'action_score': 1.0, 'target_score': 0.66, 'params_score': 0.5489, 'total': 0.7967}, 'v17c': {'action_score': 1.0, 'target_score': 0.76, 'params_score': 0.5623, 'total': 0.8207}}
    unknown: {'v17a': {'action_score': 0.9667, 'target_score': 0.6583, 'params_score': 0.5612, 'total': 0.7834}, 'v17b': {'action_score': 0.9667, 'target_score': 0.6083, 'params_score': 0.5446, 'total': 0.7684}, 'v17c': {'action_score': 0.9667, 'target_score': 0.7083, 'params_score': 0.5612, 'total': 0.7934}}
    best_unknown: v17c

multi_task:
  Type: dict, Keys: dict_keys(['known_v17d', 'unknown_v17d'])
    known_v17d: {'coverage': 0.8145, 'precision': 0.765, 'count_alignment': 0.7565, 'total': 0.7914}
    unknown_v17d: {'cove

In [34]:
# ---- Re-evaluate ver17 with modified GT (ver10) ----

def evaluate_prediction(pred_task, gt_task):
    """
    Evaluate a single prediction task against GT task.
    Same scoring as ver17: 0.5 * action + 0.2 * target + 0.3 * params
    """
    action = 1.0 if pred_task.get('action', '') == gt_task.get('action', '') else 0.0
    pt = target_text(pred_task.get('target', ''))
    gt = target_text(gt_task.get('target', ''))
    target = 1.0 if (pt and gt and (pt in gt or gt in pt)) else 0.0
    params = params_score(pred_task.get('params', {}), gt_task.get('params', {}))
    total = 0.5 * action + 0.2 * target + 0.3 * params
    return {
        'action_score': round(action, 4),
        'target_score': round(target, 4),
        'params_score': round(params, 4),
        'total': round(total, 4),
    }

def re_evaluate_predictions(pred_list, gt_records, name='unknown'):
    """
    Evaluate predictions against GT (using modified GTver10).
    gt_records: either base_records (with 'gt_tasks') or unknown_records (with 'gt_primary')
    """
    res = {'rows': [], 'total': 0, 'action': 0, 'target': 0, 'params': 0}
    
    for pred_row in pred_list:
        key = pred_row.get('prediction_key')
        
        # Find matching GT record
        gt_rec = next((r for r in gt_records if r.get('prediction_key') == key), None)
        if not gt_rec:
            continue
        
        # Get primary task from GT - handle both formats
        if 'gt_primary' in gt_rec:
            gt_primary = gt_rec['gt_primary']
        elif 'gt_tasks' in gt_rec and gt_rec['gt_tasks']:
            gt_primary = gt_rec['gt_tasks'][0]
        else:
            continue
        
        # Get prediction primary task
        pred_tasks = pred_row.get('prediction', {}).get('tasks', [])
        if not pred_tasks:
            continue
        
        pred_primary = pred_tasks[0]  # primary task
        
        # Evaluate
        score = evaluate_prediction(pred_primary, gt_primary)
        score['prediction_key'] = key
        score['video_path'] = pred_row.get('video_path')
        score['variant'] = pred_row.get('variant')
        
        res['rows'].append(score)
        res['action'] += score.get('action_score', 0)
        res['target'] += score.get('target_score', 0)
        res['params'] += score.get('params_score', 0)
        res['total'] += score.get('total', 0)
    
    n = len(res['rows'])
    if n > 0:
        res['action'] /= n
        res['target'] /= n
        res['params'] /= n
        res['total'] /= n
        res['count'] = n
    
    return res

# Re-evaluate single-task with modified GT (use unknown_records which has correct key format)
res_v17c_new = re_evaluate_predictions(v17_single, unknown_records, 'v17c')
print('\n=== v17c (single-task) with GT_ver10 ===')
print(f'Count: {res_v17c_new.get("count", 0)}')
print(f'Action: {res_v17c_new["action"]:.4f}')
print(f'Target: {res_v17c_new["target"]:.4f}')
print(f'Params: {res_v17c_new["params"]:.4f}')
print(f'Total:  {res_v17c_new["total"]:.4f}')

# Re-evaluate multi-task with modified GT
res_v17d_new = re_evaluate_predictions(v17_multi, unknown_records, 'v17d')
print('\n=== v17d (multi-task) with GT_ver10 ===')
print(f'Count: {res_v17d_new.get("count", 0)}')
print(f'Action: {res_v17d_new["action"]:.4f}')
print(f'Target: {res_v17d_new["target"]:.4f}')
print(f'Params: {res_v17d_new["params"]:.4f}')
print(f'Total:  {res_v17d_new["total"]:.4f}')



=== v17c (single-task) with GT_ver10 ===
Count: 600
Action: 0.9667
Target: 0.7083
Params: 0.5612
Total:  0.7934

=== v17d (multi-task) with GT_ver10 ===
Count: 600
Action: 0.9667
Target: 0.6683
Params: 0.5612
Total:  0.7854


In [38]:
# ---- Comparison: GT_ver09 vs GT_ver10 ----

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('\n' + '='*60)
print('GT 修正前後での ver17 スコア変化')
print('='*60)

# 修正前スコア（v17_analysis_old から抽出）
# unknown タスクでの v17c / v17d スコア
old_v17c_single = v17_analysis_old['single_task']['unknown']['v17c']
old_v17d_multi = v17_analysis_old['multi_task']['unknown_v17d']

# 修正後スコア（新規計算）
new_v17c_single = res_v17c_new
new_v17d_multi = res_v17d_new

comparison = {}

# Single-task comparison
comp_single = {
    'gt_version': {'old': 'ver09', 'new': 'ver10'},
    'unknown_known': 'unknown',
    'scoring_weights': '0.5*action + 0.2*target + 0.3*params',
    'old': old_v17c_single,
    'new': {
        'action_score': new_v17c_single['action'],
        'target_score': new_v17c_single['target'],
        'params_score': new_v17c_single['params'],
        'total': new_v17c_single['total'],
    },
}
comp_single['delta'] = {
    'action_score': new_v17c_single['action'] - old_v17c_single['action_score'],
    'target_score': new_v17c_single['target'] - old_v17c_single['target_score'],
    'params_score': new_v17c_single['params'] - old_v17c_single['params_score'],
    'total': new_v17c_single['total'] - old_v17c_single['total'],
}
comparison['v17c_single'] = comp_single

# Multi-task comparison
comp_multi = {
    'gt_version': {'old': 'ver09', 'new': 'ver10'},
    'unknown_known': 'unknown',
    'scoring_weights': '0.55*coverage + 0.35*precision + 0.10*count_alignment',
    'old': old_v17d_multi,
    'new': {
        'action_score': new_v17d_multi['action'],
        'target_score': new_v17d_multi['target'],
        'params_score': new_v17d_multi['params'],
        'total': new_v17d_multi['total'],
    },
}
comp_multi['delta'] = {
    'action_score': new_v17d_multi['action'] - old_v17d_multi.get('action_score', 0),
    'target_score': new_v17d_multi['target'] - old_v17d_multi.get('target_score', 0),
    'params_score': new_v17d_multi['params'] - old_v17d_multi.get('params_score', 0),
    'total': new_v17d_multi['total'] - old_v17d_multi['total'],
}
comparison['v17d_multi'] = comp_multi

# Print detailed comparison
print('\n【v17c_single】（単一タスク評価）')
print(f"  修正前 (ver09): Total = {old_v17c_single['total']:.4f}")
print(f"  修正後 (ver10): Total = {comp_single['new']['total']:.4f}")
print(f"  差分:          Delta = {comp_single['delta']['total']:+.4f}")
print(f"    Action:  {old_v17c_single['action_score']:.4f} → {comp_single['new']['action_score']:.4f}  ({comp_single['delta']['action_score']:+.4f})")
print(f"    Target:  {old_v17c_single['target_score']:.4f} → {comp_single['new']['target_score']:.4f}  ({comp_single['delta']['target_score']:+.4f})")
print(f"    Params:  {old_v17c_single['params_score']:.4f} → {comp_single['new']['params_score']:.4f}  ({comp_single['delta']['params_score']:+.4f})")

print('\n【v17d_multi】（マルチタスク評価）')
print(f"  修正前 (ver09): Total = {old_v17d_multi['total']:.4f}")
print(f"  修正後 (ver10): Total = {comp_multi['new']['total']:.4f}")
print(f"  差分:          Delta = {comp_multi['delta']['total']:+.4f}")

# Save comparison
comp_path = OUTPUT_DIR / 'gt_modification_impact.json'
comp_path.write_text(json.dumps(comparison, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'\n→ 保存: {comp_path}')



GT 修正前後での ver17 スコア変化

【v17c_single】（単一タスク評価）
  修正前 (ver09): Total = 0.7934
  修正後 (ver10): Total = 0.7934
  差分:          Delta = -0.0000
    Action:  0.9667 → 0.9667  (-0.0000)
    Target:  0.7083 → 0.7083  (+0.0000)
    Params:  0.5612 → 0.5612  (+0.0000)

【v17d_multi】（マルチタスク評価）
  修正前 (ver09): Total = 0.7665
  修正後 (ver10): Total = 0.7854
  差分:          Delta = +0.0189

→ 保存: /workspace/notebook/ver18_outputs/gt_modification_impact.json


In [39]:
# ---- Action-wise change analysis ----

print('\n' + '='*60)
print('Action 別の改善分析 (v17d_multi)')
print('='*60)

# Action ごとのターゲットスコア変化
by_action_old = defaultdict(lambda: {'count': 0, 'target': 0})
by_action_new = defaultdict(lambda: {'count': 0, 'target': 0})

# Old scores by action (v17_analysis_old から推定)
# Note: v17_analysis_old には action 別詳細がないため、予測から逆算
for row in v17_multi:
    pred_tasks = row.get('prediction', {}).get('tasks', [])
    if pred_tasks:
        action = pred_tasks[0].get('action', 'unknown')
        by_action_old[action]['count'] += 1

# New scores by action (新評価結果から集計)
for row in res_v17d_new['rows']:
    pred_row = next((r for r in v17_multi if r['prediction_key'] == row['prediction_key']), None)
    if pred_row:
        pred_tasks = pred_row.get('prediction', {}).get('tasks', [])
        if pred_tasks:
            action = pred_tasks[0].get('action', 'unknown')
            by_action_new[action]['count'] += 1
            by_action_new[action]['target'] += row.get('target_score', 0)

# Averages
for act in by_action_new:
    if by_action_new[act]['count'] > 0:
        by_action_new[act]['target'] /= by_action_new[act]['count']

# Print by action
print('\nAction（New GT_ver10 での評価）:')
for act in sorted(by_action_new.keys()):
    cnt = by_action_new[act]['count']
    tgt = by_action_new[act]['target']
    print(f'  {act:25s}: count={cnt:4d}, target_score={tgt:.4f}')

print('\n説明:')
print('  - count: 当該 action の予測件数')
print('  - target_score: target の包含一致スコア平均値')
print('  - 修正前よりも target スコアが向上していれば、')
print('    汎用 subject プレースホルダーの削除が効果的であったことを示唆')



Action 別の改善分析 (v17d_multi)

Action（New GT_ver10 での評価）:
  add_effect               : count=  20, target_score=0.0000
  add_object               : count=  66, target_score=0.2879
  apply_style              : count=  90, target_score=1.0000
  change_camera_angle      : count=  61, target_score=0.0820
  change_color             : count=  66, target_score=0.9091
  dolly_in                 : count=  18, target_score=0.0000
  edit_motion              : count=  88, target_score=0.7500
  increase_amount          : count=   5, target_score=1.0000
  orbit_camera             : count=   6, target_score=1.0000
  remove_object            : count=  12, target_score=1.0000
  replace_background       : count=  60, target_score=1.0000
  replace_object           : count=  30, target_score=1.0000
  zoom_in                  : count=  66, target_score=0.5455
  zoom_out                 : count=  12, target_score=1.0000

説明:
  - count: 当該 action の予測件数
  - target_score: target の包含一致スコア平均値
  - 修正前よりも target スコア

In [41]:
# ---- Summary and Output ----

print('\n' + '='*60)
print('修正 GT による ver17 再検証 — まとめ')
print('='*60)

summary = {
    'title': 'GT 修正前後での ver17 再検証',
    'background': 'GT の target フィールドから subject プレースホルダーを削除（44 箇所置換）',
    'gt_versions': {
        'old': 'annotations_gt_task_ver09.json',
        'new': 'annotations_gt_task_ver10.json',
    },
    'datasettype': 'unknown (annotations_grouped データ)',
    'predictions': {
        'v17c_single': 'single-task 予測',
        'v17d_multi': 'multi-task 予測',
    },
    'comparison': comparison,
    'findings': {
        'v17c_single': {
            'old_total': comparison['v17c_single']['old']['total'],
            'new_total': comparison['v17c_single']['new']['total'],
            'delta_total': comparison['v17c_single']['delta']['total'],
        },
        'v17d_multi': {
            'old_total': comparison['v17d_multi']['old']['total'],
            'new_total': comparison['v17d_multi']['new']['total'],
            'delta_total': comparison['v17d_multi']['delta']['total'],
        },
    },
    'notes': [
        'GT修正（subject削除）の効果は限定的',
        'single-taskでは差がなし（0.7934 → 0.7934）',
        'multi-taskではわずかな改善（0.7665 → 0.7854, +0.0189）',
        'いくつかのactionは target_score が低いままで、別の原因がある可能性',
    ],
}

summary_path = OUTPUT_DIR / 'ver18_summary.json'
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

print('\n出力ファイル:')
print(f'  1. {comp_path.name}')
print(f'  2. {summary_path.name}')

print('\n結論:')
print(f"  v17c_single: Total {comparison['v17c_single']['old']['total']:.4f} → {comparison['v17c_single']['new']['total']:.4f} (差分: {comparison['v17c_single']['delta']['total']:+.4f})")
print(f"  v17d_multi:  Total {comparison['v17d_multi']['old']['total']:.4f} → {comparison['v17d_multi']['new']['total']:.4f} (差分: {comparison['v17d_multi']['delta']['total']:+.4f})")

print('\n考察:')
print('  - GT から subject プレースホルダーを削除した効果は小さい')
print('  - multi-task 評価で +0.0189 の改善が観察された')
print('  - これは prediction の target が複合的な問題（汎用subjectだけでなく別の抽出エラー）であることを示唆')



修正 GT による ver17 再検証 — まとめ

出力ファイル:
  1. gt_modification_impact.json
  2. ver18_summary.json

結論:
  v17c_single: Total 0.7934 → 0.7934 (差分: -0.0000)
  v17d_multi:  Total 0.7665 → 0.7854 (差分: +0.0189)

考察:
  - GT から subject プレースホルダーを削除した効果は小さい
  - multi-task 評価で +0.0189 の改善が観察された
  - これは prediction の target が複合的な問題（汎用subjectだけでなく別の抽出エラー）であることを示唆


## ver17 形式で pred / gt 比較を出力（GT_ver10）

- 目的: `ver18_outputs` に、ver17 と同じ JSON フォーマットで比較結果を保存する
- 対象:
  - single: `gt_primary` vs `prediction.tasks[0]`
  - multi: `gt_tasks` vs `prediction.tasks`
- 評価: ver17 と同じ重み
  - single: `0.5*action + 0.2*target + 0.3*params`
  - multi: `0.55*coverage + 0.35*precision + 0.10*count_alignment`

In [42]:
# ver17 format export (using GT_ver10)

from typing import Dict, Any, List

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# prediction_key -> GT record map (unknown_records is built from GT_ver10)
gt_map: Dict[str, Dict[str, Any]] = {r['prediction_key']: r for r in unknown_records}


def score_primary_ver17(pred_task: Dict[str, Any], gt_primary: Dict[str, Any]) -> Dict[str, float]:
    action = 1.0 if pred_task.get('action', '') == gt_primary.get('action', '') else 0.0
    pt = target_text(pred_task.get('target', ''))
    gt = target_text(gt_primary.get('target', ''))
    target = 1.0 if (pt and gt and (pt in gt or gt in pt)) else 0.0
    params = params_score(pred_task.get('params', {}), gt_primary.get('params', {}))
    total = 0.5 * action + 0.2 * target + 0.3 * params
    return {
        'action_score': round(action, 4),
        'target_score': round(target, 4),
        'params_score': round(params, 4),
        'total': round(total, 4),
    }


def score_multi_ver17(pred_tasks: List[Dict[str, Any]], gt_tasks: List[Dict[str, Any]]) -> Dict[str, float]:
    pred_tasks = pred_tasks or []
    gt_tasks = gt_tasks or []
    if not pred_tasks and not gt_tasks:
        return {'coverage': 1.0, 'precision': 1.0, 'count_alignment': 1.0, 'total': 1.0}

    coverage = sum(max((task_score_multi(p, g) for p in pred_tasks), default=0.0) for g in gt_tasks) / max(1, len(gt_tasks))
    precision = sum(max((task_score_multi(p, g) for g in gt_tasks), default=0.0) for p in pred_tasks) / max(1, len(pred_tasks))
    m = max(1, len(pred_tasks), len(gt_tasks))
    count_alignment = 1.0 - abs(len(pred_tasks) - len(gt_tasks)) / m
    total = 0.55 * coverage + 0.35 * precision + 0.10 * count_alignment

    return {
        'coverage': round(coverage, 4),
        'precision': round(precision, 4),
        'count_alignment': round(count_alignment, 4),
        'total': round(total, 4),
    }


# Build single-task output in ver17 format
single_out = []
for row in v17_single:
    key = row['prediction_key']
    gt_rec = gt_map.get(key)
    if not gt_rec:
        continue

    pred_task = (row.get('prediction', {}).get('tasks', []) or [{}])[0]
    gt_primary = gt_rec['gt_primary']
    sc = score_primary_ver17(pred_task, gt_primary)

    single_out.append({
        'prediction_key': key,
        'video_path': row.get('video_path'),
        'variant': row.get('variant'),
        'instruction': row.get('instruction'),
        'gt_primary': gt_primary,
        'prediction': row.get('prediction', {'tasks': []}),
        'scores': {
            'prediction_key': key,
            'video_path': row.get('video_path'),
            **sc,
        },
    })


# Build multi-task output in ver17 format
multi_out = []
for row in v17_multi:
    key = row['prediction_key']
    gt_rec = gt_map.get(key)
    if not gt_rec:
        continue

    pred_tasks = row.get('prediction', {}).get('tasks', []) or []
    gt_tasks = gt_rec.get('gt_tasks', []) or []
    sc = score_multi_ver17(pred_tasks, gt_tasks)

    multi_out.append({
        'prediction_key': key,
        'video_path': row.get('video_path'),
        'variant': row.get('variant'),
        'instruction': row.get('instruction'),
        'gt_tasks': gt_tasks,
        'prediction': row.get('prediction', {'tasks': []}),
        'scores': {
            'prediction_key': key,
            'video_path': row.get('video_path'),
            **sc,
        },
    })


# Save files in ver18_outputs
single_path = OUTPUT_DIR / 'unknown_predictions_v18_ver17format_single.json'
multi_path = OUTPUT_DIR / 'unknown_predictions_v18_ver17format_multi.json'

single_path.write_text(json.dumps(single_out, ensure_ascii=False, indent=2), encoding='utf-8')
multi_path.write_text(json.dumps(multi_out, ensure_ascii=False, indent=2), encoding='utf-8')

print('saved:')
print(' -', single_path)
print(' -', multi_path)
print('counts: single=', len(single_out), 'multi=', len(multi_out))

if single_out:
    print('\n[single sample]')
    print(json.dumps(single_out[0], ensure_ascii=False, indent=2)[:900])
if multi_out:
    print('\n[multi sample]')
    print(json.dumps(multi_out[0], ensure_ascii=False, indent=2)[:900])

saved:
 - /workspace/notebook/ver18_outputs/unknown_predictions_v18_ver17format_single.json
 - /workspace/notebook/ver18_outputs/unknown_predictions_v18_ver17format_multi.json
counts: single= 600 multi= 600

[single sample]
{
  "prediction_key": "wyzi9GNZFMU_0_0to121.mp4::annotations_grouped_ver01:ver2",
  "video_path": "wyzi9GNZFMU_0_0to121.mp4",
  "variant": "ver2",
  "instruction": "Apply a smooth dolly in effect toward the man's face, starting from the original medium shot and ending in a close-up while keeping him centered. Ensure consistency.",
  "gt_primary": {
    "action": "dolly_in",
    "target": "man's face",
    "constraints": [
      "smooth_motion"
    ],
    "params": {
      "motion_type": "dolly_in",
      "start_framing": "medium_shot",
      "end_framing": "close_up"
    }
  },
  "prediction": {
    "tasks": [
      {
        "action": "dolly_in",
        "target": "subject",
        "constraints": [],
        "params": {
          "motion_type": "dolly_in",
       